<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day12-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 12, Segment 3 discussion — Does ESM-2 recognize orthologs across species?

The book page compares three *paralogous* human globins (HBB, myoglobin, HBA1) from
within one genome. This notebook asks a different, harder question: does ESM-2 also
place two *orthologous* copies of the same protein from two very different species —
human and sperm whale myoglobin — closer together than either is to an unrelated protein,
using real sequences fetched live from UniProt?

In [1]:
import urllib.request

def fetch_uniprot_fasta(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    with urllib.request.urlopen(url, timeout=15) as response:
        text = response.read().decode()
    lines = text.strip().split("\n")
    header = lines[0]
    sequence = "".join(lines[1:])
    return header, sequence

accessions = {
    "Myoglobin (human, P02144)": "P02144",
    "Myoglobin (sperm whale, P02185)": "P02185",
    "Lysozyme C (human, P61626, unrelated)": "P61626",
}

sequences = {}
for label, acc in accessions.items():
    header, seq = fetch_uniprot_fasta(acc)
    sequences[label] = seq
    print(f"{label}: {len(seq)} residues -- {header[:70]}")

Myoglobin (human, P02144): 154 residues -- >sp|P02144|MYG_HUMAN Myoglobin OS=Homo sapiens OX=9606 GN=MB PE=1 SV=2
Myoglobin (sperm whale, P02185): 154 residues -- >sp|P02185|MYG_PHYMC Myoglobin OS=Physeter macrocephalus OX=9755 GN=MB


Lysozyme C (human, P61626, unrelated): 148 residues -- >sp|P61626|LYSC_HUMAN Lysozyme C OS=Homo sapiens OX=9606 GN=LYZ PE=1 S


In [2]:
import torch
from transformers import AutoTokenizer, EsmModel

tokenizer_esm = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm_model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm_model.eval()

names = list(sequences.keys())
seqs = list(sequences.values())

with torch.no_grad():
    enc = tokenizer_esm(seqs, return_tensors="pt", padding=True)
    out = esm_model(**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)

esm_embeddings = pooled.numpy()

import numpy as np
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("ESM-2 embedding cosine similarity:")
for i in range(3):
    for j in range(i + 1, 3):
        print(f"  {names[i]:38s} vs {names[j]:38s}: {cosine(esm_embeddings[i], esm_embeddings[j]):.4f}")

/home/arnee/miniforge3/envs/ekman-teaching/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/107 [00:00<00:00, 18477.11it/s, Materializing param=contact_head.regression.bias]

Loading weights:   1%|          | 1/107 [00:00<00:00, 564.66it/s, Materializing param=contact_head.regression.bias]  

Loading weights:   2%|▏         | 2/107 [00:00<00:00, 446.84it/s, Materializing param=contact_head.regression.weight]

...

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 822.86it/s, Materializing param=encoder.layer.5.output.dense.weight]

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 822.86it/s, Materializing param=encoder.layer.5.output.dense.weight]

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 900.74it/s, Materializing param=encoder.layer.5.output.dense.weight]


EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 embedding cosine similarity:
  Myoglobin (human, P02144)              vs Myoglobin (sperm whale, P02185)       : 0.9946
  Myoglobin (human, P02144)              vs Lysozyme C (human, P61626, unrelated) : 0.8830
  Myoglobin (sperm whale, P02185)        vs Lysozyme C (human, P61626, unrelated) : 0.8833


In [3]:
from Bio import Align

aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5
aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")

def pct_identity(a, b):
    aln = aligner.align(a, b)[0]
    s1, s2 = str(aln[0]), str(aln[1])
    matches = sum(1 for x, y in zip(s1, s2) if x == y and x != "-")
    aligned_len = sum(1 for x, y in zip(s1, s2) if x != "-" and y != "-")
    return 100.0 * matches / aligned_len

print("Real pairwise sequence identity (BLOSUM62 global alignment):")
for i in range(3):
    for j in range(i + 1, 3):
        print(f"  {names[i]:38s} vs {names[j]:38s}: {pct_identity(seqs[i], seqs[j]):.1f}% identity")

Real pairwise sequence identity (BLOSUM62 global alignment):
  Myoglobin (human, P02144)              vs Myoglobin (sperm whale, P02185)       : 84.4% identity
  Myoglobin (human, P02144)              vs Lysozyme C (human, P61626, unrelated) : 24.6% identity
  Myoglobin (sperm whale, P02185)        vs Lysozyme C (human, P61626, unrelated) : 27.2% identity


## Try it yourself

Pick a different ortholog pair (e.g. human and mouse hemoglobin) and an unrelated protein
of your choice, fetch them live from UniProt the same way, and check whether the same
pattern holds: does the orthologous pair land closest together in both real sequence
identity *and* ESM-2 embedding space, the same way the book page's paralogous globins did?